In [2]:
import tensorflow as tf
import pickle
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np 

In [4]:
with open('label_encoder_gen.pkl' , 'rb') as file:
      label_encoder_gen =pickle.load(file)
with open('onehot_encoder_geo.pkl' , 'rb') as file:
      onehot_encoder_geo = pickle.load(file)  

with open('scalar.pkl' , 'rb') as file:
      scalar = pickle.load(file)   

model = load_model('model_h5')

In [52]:
# Demo Data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [53]:
# data preprocessing
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded , columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [58]:
# covert the object into table
input_df = pd.DataFrame([input_data] )
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [59]:
input_df['Gender'] = label_encoder_gen.fit_transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,0,40,3,60000,2,1,1,50000


In [60]:
# concat the geo_encoded_df to the main data
input_df = pd.concat([input_df.drop('Geography', axis=1) , geo_encoded_df] , axis=1)


In [61]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
input_scalar = scalar.transform(input_df)
input_scalar # this is the scalar data for the input

array([[-0.53598516, -1.09499335,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [65]:
# pass the data to the model to get prediction
prediction = model.predict(input_scalar)
prediction

1/1 [==============================] - 0s 18ms/step


array([[0.05467543]], dtype=float32)

In [66]:
if(prediction > 0.5):
      print("Customer is likely to churn")
else:
      print("Customer  is not likely to churn")      

Customer  is not likely to churn
